# Détection satellite — inondation Lettonie occidentale (EMSR926, août 2026)

**Événement réel** : activation Copernicus EMS EMSR926, tempête + crue le 21-22/08/2026 en Lettonie occidentale (bassin de la Venta), alerte rouge.

**Méthode** : détection par radar (Sentinel-1 SAR), pas par optique (Sentinel-2). Pourquoi le radar ici et pas l'optique ?
- L'événement est une tempête active : ciel couvert de nuages au moment où ça compte. L'optique (Sentinel-2) ne voit rien à travers les nuages.
- Le radar (Sentinel-1) envoie son propre signal et mesure ce qui lui revient (rétrodiffusion / *backscatter*) — traverse les nuages, fonctionne de nuit comme de jour.

**Principe physique de la détection** : une surface d'eau calme (rivière en crue, champ inondé) renvoie très peu de signal radar vers le satellite (comme un miroir qui dévie le rayon ailleurs) → elle apparaît **sombre** en imagerie SAR. Une surface sèche (végétation, bâti, sol nu) renvoie beaucoup plus de signal → elle apparaît **claire**. Donc : on compare une image *avant* l'événement à une image *après*, on cherche les zones qui deviennent brutalement sombres → ce sont les zones nouvellement inondées.

**Zone d'intérêt (AOI)** : approximation manuelle autour du bassin de la Venta (Lettonie occidentale, zone d'alerte rouge citée par Copernicus EMS) — le détail précis des 4 AOIs officielles EMS est réservé aux autorités locales (accès connecté requis), donc on part sur une zone raisonnable qu'on pourra affiner.

In [1]:
import ee
import geemap

PROJECT_ID = "zeta-bonfire-478712-n9"
ee.Initialize(project=PROJECT_ID)  # le token est déjà en cache depuis le notebook précédent, pas besoin de ré-authentifier

print("GEE prêt")

GEE prêt


In [2]:
# Zone d'intérêt : rectangle englobant le bassin de la Venta, Lettonie occidentale
# (Kuldīga / Saldus / Ventspils) — à affiner une fois qu'on voit les premiers résultats
aoi = ee.Geometry.Rectangle([21.4, 56.6, 22.6, 57.3])

Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.addLayer(aoi, {"color": "red"}, "Zone d'intérêt (approximative)")
Map

Map(center=[56.95034070951445, 21.999999999999822], controls=(WidgetControl(options=['position', 'transparent_…

## Charger les images Sentinel-1 avant / après

On filtre :
- sur la zone d'intérêt,
- sur le mode d'acquisition standard (`IW`, le mode par défaut sur terre),
- sur la polarisation `VV` (la plus fiable pour distinguer eau/non-eau),
- puis sur deux fenêtres de dates : une **avant** l'événement (avant le 21/08), une **après** (à partir du 22/08, la plus récente disponible).

In [3]:
from datetime import datetime, timedelta, timezone


def estimer_prochain_passage(aoi, depuis="2026-06-01"):
    """Estime la date du prochain passage Sentinel-1 sur l'AOI à partir du cycle de revisite observé récemment."""
    hist = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filterDate(depuis, datetime.now(timezone.utc).strftime("%Y-%m-%d"))
        .sort("system:time_start")
    )
    timestamps = hist.aggregate_array("system:time_start").getInfo()
    dates_uniques = sorted(
        {datetime.fromtimestamp(t / 1000, tz=timezone.utc).date() for t in timestamps}
    )
    if len(dates_uniques) < 2:
        return None, None

    # Regroupe les dates en "grappes" de passages rapprochés (< 2 jours d'écart entre deux passages)
    grappes = [[dates_uniques[0]]]
    for d in dates_uniques[1:]:
        if (d - grappes[-1][-1]).days <= 2:
            grappes[-1].append(d)
        else:
            grappes.append([d])
    debuts_grappes = [g[0] for g in grappes]

    ecarts = [
        (debuts_grappes[i + 1] - debuts_grappes[i]).days
        for i in range(len(debuts_grappes) - 1)
    ]
    if not ecarts:
        return None, None
    periode = sorted(ecarts)[len(ecarts) // 2]  # médiane : robuste si un écart est atypique

    prochain = debuts_grappes[-1]
    aujourdhui = datetime.now(timezone.utc).date()
    while prochain <= aujourdhui:
        prochain += timedelta(days=periode)

    return prochain, periode


s1 = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(aoi)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
    .select("VV")
)

avant = s1.filterDate("2026-08-01", "2026-08-21")
apres = s1.filterDate("2026-08-22", "2026-08-25")

n_avant = avant.size().getInfo()
n_apres = apres.size().getInfo()

print(f"Images disponibles avant l'événement : {n_avant}")
print(f"Images disponibles après l'événement  : {n_apres}")

# Le pipeline ne doit pas planter si le satellite n'est pas encore repassé : c'est un état
# normal et fréquent d'un système réactif, pas une erreur. On le détecte et on informe.
donnees_pretes = n_apres > 0

if not donnees_pretes:
    prochain, periode = estimer_prochain_passage(aoi)
    print()
    if prochain:
        print(f"Aucune image post-événement disponible pour l'instant.")
        print(f"Cycle de revisite Sentinel-1 observé sur cette zone : ~{periode} jours.")
        print(f"Prochain passage estimé sur la zone : {prochain}")
        print("→ Relancer ce notebook à partir de cette date pour obtenir la détection.")
    else:
        print("Historique de passages insuffisant pour estimer la prochaine date.")

Images disponibles avant l'événement : 29
Images disponibles après l'événement  : 0



Aucune image post-événement disponible pour l'instant.
Cycle de revisite Sentinel-1 observé sur cette zone : ~6 jours.
Prochain passage estimé sur la zone : 2026-08-30
→ Relancer ce notebook à partir de cette date pour obtenir la détection.


Si l'un des deux compteurs affiche 0, c'est probablement qu'aucun passage Sentinel-1 n'a encore survolé la zone dans cette fenêtre (le satellite ne repasse pas au même endroit tous les jours, cycle ~6 jours par satellite). Dans ce cas, il faudra élargir légèrement les dates plutôt que forcer un résultat.

In [4]:
if donnees_pretes:
    # Composite médian sur chaque période (réduit le bruit "speckle" typique du radar)
    img_avant = avant.median().clip(aoi)
    img_apres = apres.median().clip(aoi)

    vis_params = {"min": -25, "max": 0}  # échelle typique du VV en décibels

    Map = geemap.Map()
    Map.centerObject(aoi, 9)
    Map.addLayer(img_avant, vis_params, "Avant (VV, dB)")
    Map.addLayer(img_apres, vis_params, "Après (VV, dB)")
    Map
else:
    print("En attente du prochain passage satellite (voir estimation ci-dessus) — rien à afficher pour l'instant.")

En attente du prochain passage satellite (voir estimation ci-dessus) — rien à afficher pour l'instant.


## Calculer le masque d'inondation

Différence *après - avant* en décibels. Une chute brutale (valeur très négative) = zone qui est devenue beaucoup moins réfléchissante = probablement de l'eau nouvellement présente. On applique un seuil (à ajuster empiriquement) pour transformer ça en masque binaire inondé / non-inondé.

In [5]:
if donnees_pretes:
    difference = img_apres.subtract(img_avant)

    SEUIL_DB = -3  # chute d'au moins 3 dB : point de départ à ajuster selon ce qu'on observe
    masque_inondation = difference.lt(SEUIL_DB).selfMask()

    Map = geemap.Map()
    Map.centerObject(aoi, 9)
    Map.addLayer(img_apres, vis_params, "Après (VV, dB)")
    Map.addLayer(masque_inondation, {"palette": ["blue"]}, "Zone probablement inondée")
    Map
else:
    print("En attente du prochain passage satellite (voir estimation ci-dessus) — rien à calculer pour l'instant.")

En attente du prochain passage satellite (voir estimation ci-dessus) — rien à calculer pour l'instant.
